#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

# Код домашней работы:

In [1]:
# Установка необходимых библиотек
!pip install transformers datasets evaluate accelerate -q

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [6]:
# 1. Загрузка датасета
print("Загрузка датасета ag_news...")
# Загружаем датасет классификации новостей (4 категории: World, Sports, Business, Sci/Tech)
dataset = load_dataset("fancyzhx/ag_news")

Загрузка датасета ag_news...


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [7]:
# 2. Выбор модели и токенизатора
# Выбираем distilbert-base-uncased (он быстрее обучается, чем обычный bert)
model_name = "distilbert-base-uncased"
print(f"Загрузка токенизатора и модели {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Указываем num_labels=4, так как у нас 4 класса новостей
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

Загрузка токенизатора и модели distilbert-base-uncased...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


В самой последней строчке лога библиотека подсказывает: "Consider training on your downstream task"

Именно этим и займётся наш класс Trainer. Во время обучения модель обновит эти случайно инициализированные веса классификатора и подстроит их под 4 категории новостей из датасета ag_news.

In [8]:
# 3. Токенизация данных
def tokenize_function(examples):
    # Токенизируем с ограничением длины до 128 токенов
    return tokenizer(examples["text"], truncation=True, max_length=128)

print("Токенизация текстов...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Формируем тренировочную и тестовую выборки
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

Токенизация текстов...


Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [9]:
# 4. Настройка параметров обучения
# Задаем требуемые аргументы из задания
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,                  # Скорость обучения
    per_device_train_batch_size=16,      # Размер батча
    per_device_eval_batch_size=16,
    num_train_epochs=3,                  # Количество эпох
    eval_strategy="epoch",               # Оценка в конце каждой эпохи
    save_strategy="epoch",               # Сохранение в конце каждой эпохи
    load_best_model_at_end=True,         # Загрузить лучшую модель в конце
    metric_for_best_model="accuracy",    # Метрика для определения лучшей модели
    weight_decay=0.01,
)

# Загружаем метрику accuracy
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

In [11]:
# 5. Инициализация Trainer и Обучение
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

print("Начало процесса fine-tuning...")
trainer.train() # Запуск обучения

Начало процесса fine-tuning...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.199134,0.177235,0.942237
2,0.133675,0.188987,0.944868
3,0.088721,0.220060,0.946053


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=22500, training_loss=0.1527842274983724, metrics={'train_runtime': 2979.9677, 'train_samples_per_second': 120.807, 'train_steps_per_second': 7.55, 'total_flos': 8558812764910464.0, 'train_loss': 0.1527842274983724, 'epoch': 3.0})

In [12]:
# 6. Оценка на тестовой выборке
print("\nОценка качества на тестовой выборке...")
eval_results = trainer.evaluate()
print(f"Результаты оценки (Accuracy): {eval_results['eval_accuracy']:.4f}") # Вывод accuracy


Оценка качества на тестовой выборке...


Training Loss,Validation Loss,Epoch,Accuracy
0.088721,0.220060,3,0.946053


Результаты оценки (Accuracy): 0.9461


In [13]:
# 7. Сохранение модели
save_path = "./ag_news_model"
print(f"\nСохранение лучшей модели в папку {save_path}...")
trainer.save_model(save_path) # Сохранение в требуемую директорию
print("Готово!")


Сохранение лучшей модели в папку ./ag_news_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Готово!
